In [ ]:
# Cell 1: Secrets
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["COMET_API_KEY"] = userdata.get("COMET_API_KEY")


In [ ]:
# Cell 2: Setup
!git clone https://github.com/sk3feel/hidden-data-reproduction-multimodal.git /content/course_work2026 2>/dev/null || (cd /content/course_work2026 && git pull)
%cd /content/course_work2026
!pip install -q -r requirements.txt
!pip install -q transformers accelerate Pillow pandas tqdm einops timm comet_ml huggingface_hub
!pip install -q peft bitsandbytes trl qwen-vl-utils
!python src/download_from_hf.py --repo-id sk3feel/docvqa-privacy-data

from pathlib import Path

assert Path("artifacts/finetuning_generative/train_qwen2vl.jsonl").exists()


In [ ]:
# Cell 3: Imports and config
import gc
import json
import math
import os
import re
from pathlib import Path

import comet_ml
import pandas as pd
import torch
from PIL import Image, ImageFile
from huggingface_hub import HfApi, login
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration

ImageFile.LOAD_TRUNCATED_IMAGES = True

MODEL_CONFIGS = [
    {"name": "Qwen/Qwen2-VL-2B-Instruct", "tag": "qwen2b", "epochs": 5, "batch_size": 2, "checkpoint_epochs": [1, 3, 5]},
    {"name": "Qwen/Qwen2-VL-7B-Instruct", "tag": "qwen7b", "epochs": 2, "batch_size": 1, "checkpoint_epochs": [2]},
]
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 2e-4
GRADIENT_ACCUMULATION = 8
HF_MODEL_REPO = "sk3feel/docvqa-privacy-checkpoints"
TRAIN_JSONL = Path("artifacts/finetuning_generative/train_qwen2vl.jsonl")
VALIDATION_JSONL = Path("artifacts/finetuning_generative/validation_qwen2vl.jsonl")

login(token=os.environ["HF_TOKEN"])
api = HfApi()
api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required")

COMPUTE_DTYPE = torch.bfloat16


In [ ]:
# Cell 4: Load data
QUESTION_RE = re.compile(r"Question:\s*(.*)", flags=re.IGNORECASE | re.DOTALL)


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(raw_path: str, split: str) -> Path:
    path = Path(raw_path)
    if path.exists():
        return path
    image_name = raw_path.replace("\\", "/").split("/")[-1]
    split_dir = "benchmark_train" if split == "train" else "benchmark"
    candidate = Path("artifacts") / "docqa_recovery" / split_dir / "images" / "original" / image_name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f"Image not found: {raw_path}")


def extract_question(row: dict) -> str:
    if row.get("question"):
        return str(row["question"]).strip()
    chat_messages = row.get("chat_messages") or []
    if chat_messages:
        for content_item in chat_messages[0].get("content", []):
            if content_item.get("type") == "text":
                text = str(content_item.get("text", "")).strip()
                match = QUESTION_RE.search(text)
                if match:
                    return match.group(1).strip()
                return text
    if row.get("prompt"):
        return str(row["prompt"]).strip()
    return ""


def prepare_records(path: Path, default_split: str) -> list[dict]:
    rows = load_jsonl(path)
    records = []
    for row in rows:
        split = str(row.get("split", default_split))
        record = {
            "example_id": str(row.get("example_id", "")),
            "image_path": str(resolve_image_path(str(row["image_path"]), split)),
            "question": extract_question(row),
            "answer": str(row["answer"]).strip(),
            "split": split,
        }
        records.append(record)
    return records


train_records = prepare_records(TRAIN_JSONL, "train")
validation_records = prepare_records(VALIDATION_JSONL, "validation")

assert len(train_records) == 800, f"Expected 800 train rows, got {len(train_records)}"
assert all(Path(rec["image_path"]).exists() for rec in train_records), "Missing train image paths"
assert all(Path(rec["image_path"]).exists() for rec in validation_records), "Missing validation image paths"

print({"train_records": len(train_records), "validation_records": len(validation_records)})

sample_rows = []
for rec in train_records[:3]:
    sample_rows.append(
        {
            "question": rec["question"],
            "answer": rec["answer"],
            "image_path": rec["image_path"],
        }
    )

display(pd.DataFrame(sample_rows))


In [ ]:
# Cell 5: Functions
class QwenDocDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]


def make_user_text(question: str) -> str:
    return f"Answer the question about this document image. Give only the answer value, nothing else.\nQuestion: {question}"


def build_messages(record: dict, image_for_message):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_for_message},
                {"type": "text", "text": make_user_text(record["question"])},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": record["answer"]}],
        },
    ]


def normalize_text(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


def get_model_device(model):
    return next(model.parameters()).device


def load_qwen_for_training(model_name):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
    )
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    lora = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    return model, processor


def cleanup_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")


def make_collate_fn(processor):
    def collate_fn(batch):
        prompt_texts = []
        full_texts = []
        prompt_images = []
        full_images = []

        for rec in batch:
            image = Image.open(rec["image_path"]).convert("RGB")
            prompt_messages = build_messages(rec, image)[:1]
            full_messages = build_messages(rec, image)

            prompt_texts.append(processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True))
            full_texts.append(processor.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False))

            prompt_image_inputs, _ = process_vision_info(prompt_messages)
            full_image_inputs, _ = process_vision_info(full_messages)
            prompt_images.append(prompt_image_inputs[0] if isinstance(prompt_image_inputs, list) else prompt_image_inputs)
            full_images.append(full_image_inputs[0] if isinstance(full_image_inputs, list) else full_image_inputs)

        prompt_batch = processor(text=prompt_texts, images=prompt_images, return_tensors="pt", padding=True)
        full_batch = processor(text=full_texts, images=full_images, return_tensors="pt", padding=True)

        labels = full_batch["input_ids"].clone()
        labels[full_batch["attention_mask"] == 0] = -100
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1).tolist()
        for idx, prompt_len in enumerate(prompt_lengths):
            labels[idx, : int(prompt_len)] = -100

        batch_dict = {k: v for k, v in full_batch.items()}
        batch_dict["labels"] = labels
        return batch_dict

    return collate_fn


def save_adapter_checkpoint(model, processor, ckpt_path: str):
    os.makedirs(ckpt_path, exist_ok=True)
    model.save_pretrained(ckpt_path)
    processor.save_pretrained(ckpt_path)


def run_sanity_check(model, processor, records, n=20):
    model.eval()
    correct = 0
    model_device = get_model_device(model)
    for rec in records[:n]:
        image = Image.open(rec["image_path"]).convert("RGB")
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"Answer the question about this document image. Give only the answer value, nothing else.\nQuestion: {rec['question']}"},
                ],
            }
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        image_input = image_inputs[0] if isinstance(image_inputs, list) else image_inputs
        inputs = processor(text=[text], images=[image_input], return_tensors="pt", padding=True)
        inputs = {k: v.to(model_device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=50)
        pred = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
        if normalize_text(pred) == normalize_text(rec["answer"]):
            correct += 1
    model.train()
    return correct / max(n, 1)


In [ ]:
# Cell 6+: Training loop
for config in MODEL_CONFIGS:
    tag = config["tag"]
    print(f"\n{'=' * 60}\nTraining {tag}: {config['name']}\n{'=' * 60}")

    experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
    experiment.set_name(f"{tag}-finetune")
    experiment.log_parameters(
        {
            **config,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lr": LR,
            "gradient_accumulation": GRADIENT_ACCUMULATION,
            "bf16": True,
            "gradient_checkpointing": True,
        }
    )

    model, processor = load_qwen_for_training(config["name"])
    dataset = QwenDocDataset(train_records)
    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        collate_fn=make_collate_fn(processor),
    )
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)

    global_step = 0
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        optimizer.zero_grad()
        epoch_loss = 0.0
        valid_batches = 0

        progress = tqdm(dataloader, desc=f"{tag} Epoch {epoch}/{config['epochs']}")
        for batch in progress:
            model_device = get_model_device(model)
            batch = {k: v.to(model_device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss / GRADIENT_ACCUMULATION
            if not loss.isfinite():
                print(f"Skipping non-finite batch in {tag} epoch {epoch}")
                optimizer.zero_grad()
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if (valid_batches + 1) % GRADIENT_ACCUMULATION == 0:
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1
            epoch_loss += float(loss.item()) * GRADIENT_ACCUMULATION
            valid_batches += 1
            progress.set_postfix({"loss": f"{epoch_loss / max(valid_batches, 1):.4f}"})

        if valid_batches % GRADIENT_ACCUMULATION != 0:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        avg_loss = epoch_loss / max(valid_batches, 1)
        print(f"{tag} epoch {epoch}, loss: {avg_loss:.4f}")
        experiment.log_metric("train_loss", avg_loss, epoch=epoch)

        if epoch in config["checkpoint_epochs"]:
            ckpt_path = f"checkpoints/{tag}/epoch_{epoch}"
            save_adapter_checkpoint(model, processor, ckpt_path)
            api.upload_folder(
                folder_path=ckpt_path,
                repo_id=HF_MODEL_REPO,
                path_in_repo=f"{tag}/epoch_{epoch}",
                repo_type="model",
            )
            em = run_sanity_check(model, processor, train_records, n=20)
            print(f"{tag} sanity EM@{epoch}: {em:.2f}")
            experiment.log_metric("sanity_em", em, epoch=epoch)

    seen_em = run_sanity_check(model, processor, train_records, n=100)
    unseen_em = run_sanity_check(model, processor, validation_records, n=100)
    print({"tag": tag, "final_seen_em": seen_em, "final_unseen_em": unseen_em})
    experiment.log_metric("final_seen_em", seen_em)
    experiment.log_metric("final_unseen_em", unseen_em)

    experiment.end()

    del model, processor, dataset, dataloader, optimizer
    cleanup_gpu()
